# Extra Metrics Evaluation

Computes three additional metrics for all 8 prediction methods on the same test set (year 2015, predicting 2020).

| Metric | Definition |
|--------|------------|
| **Best F1** | F1 at the decision threshold that maximises F1 |
| **Precision@1000** | Fraction of the top-1000 predicted pairs that are true positives |
| **mAP@10** | Mean Average Precision at rank 10, averaged across countries |

ECI has no product-level ranking signal (all products in a country receive the same score), so **mAP@10 is N/A** for that method.

In [1]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_recall_curve
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
warnings.filterwarnings('ignore')

DATA_DIR  = 'data'
TEST_YEAR = 2015
TRAIN_CUTOFF = 2012
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Raw data ──────────────────────────────────────────────────────────────────
smooth    = pd.read_csv(os.path.join(DATA_DIR, 'M_cpt_smoothed.csv'))
rca_df    = pd.read_csv(os.path.join(DATA_DIR, 'rca_cpt.csv'))
test_lbl  = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))
train_lbl = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))

countries = sorted(smooth['country'].unique())
products  = sorted(smooth['product'].unique())
C, P = len(countries), len(products)
c_idx = {c: i for i, c in enumerate(countries)}
p_idx = {p: i for i, p in enumerate(products)}

def build_M(year):
    M = np.zeros((C, P), dtype=np.float32)
    yr = smooth[smooth['year'] == year]
    M[yr['country'].map(c_idx).values, yr['product'].map(p_idx).values] = 1.0
    return M

M_t    = build_M(TEST_YEAR)
y_true = test_lbl['label'].values
ci_arr = test_lbl['country'].map(c_idx).values
pi_arr = test_lbl['product'].map(p_idx).values

# ── Proximity matrix (train years only) ───────────────────────────────────────
print('Building proximity matrix...')
co_exp = np.zeros((P, P), dtype=np.float32)
any_exp = np.zeros((P, P), dtype=np.float32)
for yr in sorted(y for y in smooth['year'].unique() if y <= TRAIN_CUTOFF):
    M = build_M(yr)
    co = M.T @ M
    ex = M.sum(axis=0)
    co_exp  += co
    any_exp += ex[:, None] + ex[None, :] - co
phi = np.where(any_exp > 0, co_exp / (any_exp + 1e-9), 0.0)
np.fill_diagonal(phi, 0.0)
phi_row_sum = phi.sum(axis=1)

# ── ECI/PCI ───────────────────────────────────────────────────────────────────
kc = M_t.sum(axis=1); kp = M_t.sum(axis=0)
kc_safe = np.where(kc > 0, kc, 1.0); kp_safe = np.where(kp > 0, kp, 1.0)
kc_n, kp_n = kc.astype(float), kp.astype(float)
for _ in range(20):
    kc_n = (1.0 / kc_safe) * (M_t   @ kp_n)
    kp_n = (1.0 / kp_safe) * (M_t.T @ kc_n)
eci = (kc_n - kc_n.mean()) / (kc_n.std() + 1e-9)

def minmax(x): return (x - x.min()) / (x.max() - x.min() + 1e-9)

print(f'Setup done. Countries: {C}  Products: {P}  Test pairs: {len(test_lbl)}')

Device: cuda
Building proximity matrix...
Setup done. Countries: 233  Products: 5018  Test pairs: 127531


## Metric Definitions

In [2]:
def best_f1(y_true, scores):
    """F1 at the threshold that maximises F1."""
    p, r, _ = precision_recall_curve(y_true, scores)
    denom = p + r
    f1 = np.where(denom > 0, 2 * p * r / denom, 0.0)
    return float(f1.max())

def precision_at_1000(y_true, scores):
    """Fraction of the top-1000 predictions that are true positives."""
    topk = np.argsort(scores)[::-1][:1000]
    return float(y_true[topk].sum()) / 1000.0

def map_at_10(test_lbl_df, scores):
    """
    Mean Average Precision at rank 10, per country.
    AP@10 for country c = sum_{k=1}^{10} P@k * rel(k)  /  min(R_c, 10)
    where R_c = number of positives for that country.
    Countries with 0 positives are excluded.
    """
    df = test_lbl_df.copy()
    df['score'] = scores
    ap_list = []
    for _, grp in df.groupby('country'):
        n_pos = int(grp['label'].sum())
        if n_pos == 0:
            continue
        top10 = grp.sort_values('score', ascending=False).head(10)['label'].values
        cumtp  = np.cumsum(top10)
        ranks  = np.arange(1, len(top10) + 1)
        prec_k = cumtp / ranks
        ap     = (prec_k * top10).sum() / min(n_pos, 10)
        ap_list.append(ap)
    return float(np.mean(ap_list)) if ap_list else 0.0

# Registry: method name -> (best_f1, prec@1000, map@10, map@10_valid)
ALL_SCORES  = {}   # method -> score array
NO_MAP_AT10 = set()  # methods where mAP@10 is not meaningful

def register(name, scores, skip_map10=False):
    scores = np.array(scores, dtype=np.float64)
    ALL_SCORES[name] = scores
    if skip_map10:
        NO_MAP_AT10.add(name)
    bf1  = best_f1(y_true, scores)
    p1k  = precision_at_1000(y_true, scores)
    m10  = map_at_10(test_lbl, scores) if not skip_map10 else float('nan')
    m10s = f'{m10:.4f}' if not skip_map10 else '  N/A '
    print(f'{name:<26}  Best-F1={bf1:.4f}  P@1000={p1k:.4f}  mAP@10={m10s}')
    return bf1, p1k, m10

print('Metric functions ready.')

Metric functions ready.


## Method 1 — RCA Persistence

In [3]:
history_years = [TEST_YEAR - 2, TEST_YEAR - 1, TEST_YEAR]
rca_hist  = rca_df[rca_df['year'].isin(history_years)][['country', 'product', 'year', 'rca']]
rca_hist  = rca_hist.merge(test_lbl[['country', 'product']], on=['country', 'product'])
rca_wide  = rca_hist.pivot_table(index=['country', 'product'], columns='year', values='rca', fill_value=0)
for yr in history_years:
    if yr not in rca_wide.columns:
        rca_wide[yr] = 0
rca_wide['score'] = (rca_wide[history_years] >= 1).mean(axis=1)
persist_df = test_lbl.merge(rca_wide[['score']], on=['country', 'product'], how='left').fillna(0)
register('RCA Persistence', persist_df['score'].values)

RCA Persistence             Best-F1=0.4357  P@1000=0.5970  mAP@10=0.3660


(0.4357086056079936, 0.597, 0.36602410802078944)

## Method 2 — Product Space Density

In [4]:
dens_mat    = (M_t @ phi) / (phi_row_sum[None, :] + 1e-9)
dens_scores = dens_mat[ci_arr, pi_arr]
register('Density', dens_scores)

Density                     Best-F1=0.4205  P@1000=0.4860  mAP@10=0.3448


(0.42053884444788786, 0.486, 0.34479483976481445)

## Method 3 — ECI

ECI is a per-country score — every product in a country gets the same value, so **mAP@10 is N/A** (no within-country ranking signal).

In [5]:
register('ECI', eci[ci_arr], skip_map10=True)

ECI                         Best-F1=0.2597  P@1000=0.0930  mAP@10=  N/A 


(0.25974920572440746, 0.093, nan)

## Method 4 — ECI + Density

In [6]:
register('ECI + Density', minmax(eci[ci_arr]) + minmax(dens_mat[ci_arr, pi_arr]))

ECI + Density               Best-F1=0.4205  P@1000=0.4860  mAP@10=0.3448


(0.42053884444788786, 0.486, 0.34479483976481445)

## Method 5 — KNN on LLM Embeddings

In [7]:
EMB_PATH = os.path.join(DATA_DIR, 'product_llm_embeddings.pt')
emb = torch.load(EMB_PATH, weights_only=False, map_location='cpu').numpy()  # [P, D] unit-normed
print(f'Embeddings: {emb.shape}')

country_basket = {}
for c in test_lbl['country'].unique():
    ci = c_idx.get(c, -1)
    if ci < 0:
        continue
    exported = np.where(M_t[ci] == 1)[0]
    if len(exported) == 0:
        country_basket[c] = np.zeros(emb.shape[1])
    else:
        basket = emb[exported].mean(axis=0)
        country_basket[c] = basket / (np.linalg.norm(basket) + 1e-9)

knn_scores = np.array([
    float(emb[p_idx[p]] @ country_basket[c])
    if c in country_basket and p in p_idx else 0.0
    for c, p in zip(test_lbl['country'].values, test_lbl['product'].values)
], dtype=np.float32)

register('KNN (LLM embeddings)', knn_scores)

Embeddings: (5018, 768)
KNN (LLM embeddings)        Best-F1=0.2997  P@1000=0.4090  mAP@10=0.1644


(0.29968011811023626, 0.409, 0.16444488341059135)

## GNN Architecture & Checkpoint Loader

Shared code for loading GNN-4F, GNN-11F, and GNN-11F+LLM from saved checkpoints.

In [8]:
CKPT_DIR = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')

edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr      = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'),  weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'),  weights_only=False)
cap_ei         = torch.load(os.path.join(DATA_DIR, 'capability_edge_index.pt'), weights_only=False).long()

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

c_feat_df = pd.read_csv(os.path.join(DATA_DIR, 'country_features.csv'))
BACI_COLS = ['log_export', 'n_products', 'avg_rca', 'max_rca']
c_x_4feat = {}
for yr in sorted(c_feat_df['year'].unique()):
    yd = c_feat_df[c_feat_df['year'] == yr].copy()
    yd['idx'] = yd['country'].map(c_map['to_idx'])
    yd = yd.dropna(subset=['idx']).sort_values('idx')
    c_x_4feat[int(yr)] = torch.tensor(yd[BACI_COLS].values, dtype=torch.float32)

class _HomoGNN(nn.Module):
    def __init__(self, hidden, drop=0.3):
        super().__init__()
        self.c1 = SAGEConv(hidden, hidden); self.c2 = SAGEConv(hidden, hidden); self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)

class BipartiteEncoder(nn.Module):
    def __init__(self, c_in, hidden, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(3, hidden)
        self.gnn = to_hetero(_HomoGNN(hidden), meta)
    def forward(self, x_dict, ei_dict):
        return self.gnn({'country': self.country_lin(x_dict['country']),
                         'product': self.product_lin(x_dict['product'])}, ei_dict)

class TemporalGNN(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

class LinkPredictor(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], -1)).view(-1)

def _build_snap(year, c_x, with_cap=False):
    d = HeteroData()
    d['country'].x = c_x[year].to(DEVICE)
    d['product'].x = p_x_by_yr[year].to(DEVICE)
    ei = edge_idx_by_yr[year].long().to(DEVICE)
    d['country', 'exports',     'product'].edge_index = ei
    d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
    if with_cap:
        d['product', 'capability', 'product'].edge_index = cap_ei.to(DEVICE)
    return d

@torch.no_grad()
def gnn_scores_from_ckpt(ckpt_path, c_x, with_cap=False):
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    enc  = BipartiteEncoder(ckpt['c_in'], ckpt['hidden'], ckpt['meta']).to(DEVICE)
    mdl  = TemporalGNN(enc, ckpt['hidden']).to(DEVICE)
    pred = LinkPredictor(ckpt['hidden']).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    snaps = [_build_snap(y, c_x, with_cap) for y in range(TEST_YEAR - 4, TEST_YEAR + 1)]

    row   = test_lbl.copy()
    ci_s  = row['country'].map(c_map['to_idx'])
    pi_s  = row['product'].map(p_map['to_idx'])
    ok    = ci_s.notna() & pi_s.notna()
    ci_v  = ci_s[ok].astype(int).values
    pi_v  = pi_s[ok].astype(int).values
    ei_t  = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    gnn_df = pd.DataFrame({'country': row.loc[ok, 'country'].values,
                           'product': row.loc[ok, 'product'].values, 'score': raw})
    merged = test_lbl.merge(gnn_df[['country', 'product', 'score']],
                            on=['country', 'product'], how='left').fillna(0)
    return merged['score'].values

print('GNN helpers loaded.')
print(f'  4-feat:  {c_x_4feat[TEST_YEAR].shape}')
print(f'  11-feat: {c_x_11feat[TEST_YEAR].shape}')
print(f'  cap_ei:  {tuple(cap_ei.shape)}')

GNN helpers loaded.
  4-feat:  torch.Size([233, 4])
  11-feat: torch.Size([233, 11])
  cap_ei:  (2, 144192)


## Method 6 — GNN-4F

In [9]:
CKPT_4F = os.path.join(CKPT_DIR, 'gnn_4f.pt')
assert os.path.exists(CKPT_4F), f'Missing checkpoint: {CKPT_4F}'
print('Loading GNN-4F...')
gnn4f_scores = gnn_scores_from_ckpt(CKPT_4F, c_x_4feat, with_cap=False)
register('GNN-4F', gnn4f_scores)

Loading GNN-4F...
GNN-4F                      Best-F1=0.4702  P@1000=0.6190  mAP@10=0.2963


(0.4702028325944525, 0.619, 0.2963112796741115)

## Method 7 — GNN-11F

In [10]:
CKPT_11F = os.path.join(CKPT_DIR, 'gnn_11f.pt')
assert os.path.exists(CKPT_11F), f'Missing checkpoint: {CKPT_11F}'
print('Loading GNN-11F...')
gnn11f_scores = gnn_scores_from_ckpt(CKPT_11F, c_x_11feat, with_cap=False)
register('GNN-11F (BACI+WDI)', gnn11f_scores)

Loading GNN-11F...
GNN-11F (BACI+WDI)          Best-F1=0.4836  P@1000=0.6500  mAP@10=0.3350


(0.48361286031963757, 0.65, 0.33503377112890387)

## Method 8 — GNN-11F+LLM

In [11]:
CKPT_LLM = os.path.join(CKPT_DIR, 'gnn_11f_llm.pt')
assert os.path.exists(CKPT_LLM), f'Missing checkpoint: {CKPT_LLM}'
print('Loading GNN-11F+LLM...')
gnn_llm_scores = gnn_scores_from_ckpt(CKPT_LLM, c_x_11feat, with_cap=True)
register('GNN-11F+LLM', gnn_llm_scores)

Loading GNN-11F+LLM...
GNN-11F+LLM                 Best-F1=0.4834  P@1000=0.6690  mAP@10=0.3688


(0.4833766858291984, 0.669, 0.36879752540150773)

## Results Table

In [12]:
METHOD_ORDER = [
    'RCA Persistence', 'Density', 'ECI', 'ECI + Density',
    'KNN (LLM embeddings)', 'GNN-4F', 'GNN-11F (BACI+WDI)', 'GNN-11F+LLM'
]
methods_run = [m for m in METHOD_ORDER if m in ALL_SCORES]

rows = []
for m in methods_run:
    scores = ALL_SCORES[m]
    skip   = m in NO_MAP_AT10
    rows.append({
        'Method':     m,
        'Best F1':    round(best_f1(y_true, scores), 4),
        'Prec@1000':  round(precision_at_1000(y_true, scores), 4),
        'mAP@10':     'N/A' if skip else round(map_at_10(test_lbl, scores), 4),
    })

df_res = pd.DataFrame(rows).set_index('Method')

print('=' * 62)
print(f'  {"Method":<26} {"Best F1":>8} {"Prec@1000":>10} {"mAP@10":>8}')
print('-' * 62)
for m, row in df_res.iterrows():
    gnn = ' <--' if 'GNN' in m else ''
    m10s = f'{row["mAP@10"]:>8}' if row['mAP@10'] == 'N/A' else f'{row["mAP@10"]:>8.4f}'
    print(f'  {m:<26} {row["Best F1"]:>8.4f} {row["Prec@1000"]:>10.4f} {m10s}{gnn}')
print('=' * 62)
print('N/A = ECI has no product-level ranking (all products in a country share the same score).')

df_res.to_csv(os.path.join(DATA_DIR, 'extra_metrics_results.csv'))
print('\nSaved -> data/extra_metrics_results.csv')

  Method                      Best F1  Prec@1000   mAP@10
--------------------------------------------------------------
  RCA Persistence              0.4357     0.5970   0.3660
  Density                      0.4205     0.4860   0.3448
  ECI                          0.2597     0.0930      N/A
  ECI + Density                0.4205     0.4860   0.3448
  KNN (LLM embeddings)         0.2997     0.4090   0.1644
  GNN-4F                       0.4702     0.6190   0.2963 <--
  GNN-11F (BACI+WDI)           0.4836     0.6500   0.3350 <--
  GNN-11F+LLM                  0.4834     0.6690   0.3688 <--
N/A = ECI has no product-level ranking (all products in a country share the same score).

Saved -> data/extra_metrics_results.csv
